In [1]:
import chromadb
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer

In [ ]:
df = pd.read_parquet("../data/processed/arxiv_ml.parquet")

#df["id"] = df["id"].astype(str)

df.head()

,id,title,authors,category,text_raw,text_ml
0,704.0001,Calculation of prompt diphoton production cros...,"C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...",hep-ph,Calculation of prompt diphoton production cros...,calculation prompt diphoton production cross s...
1,704.0002,Sparsity-certifying Graph Decompositions,Ileana Streinu and Louis Theran,math.CO,Sparsity-certifying Graph Decompositions We ...,sparsity certify graph decomposition describe ...
2,704.0003,The evolution of the Earth-Moon system based o...,Hongjun Pan,physics.gen-ph,The evolution of the Earth-Moon system based o...,evolution earth moon system base dark matter f...
3,704.0004,A determinant of Stirling cycle numbers counts...,David Callan,math.CO,A determinant of Stirling cycle numbers counts...,determinant stirling cycle number count unlabe...
4,704.0005,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,Wael Abu-Shammala and Alberto Torchinsky,math.CA,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,dyadic lambda alpha lambda alpha compute lambd...


In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
embeddings = model.encode(
    df["text_raw"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(embeddings.shape)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

(1000, 384)


In [5]:
client = chromadb.Client()

In [6]:
import chromadb

client = chromadb.Client()

print(client)

In [7]:
collection = client.create_collection(
    name="research_papers"
)

print(collection)

Collection(name=research_papers)


In [8]:
client.list_collections()

[Collection(name=research_papers)]

In [9]:
print(type(df["id"].iloc[0]))
print(df["id"].iloc[0])

<class 'numpy.float64'>
704.0001


In [10]:
print(type(embeddings[0]))
print(embeddings[0].shape)

<class 'numpy.ndarray'>
(384,)


In [11]:
print(type(embeddings[0]))
print(embeddings[0].shape)

<class 'numpy.ndarray'>
(384,)


In [12]:
print(df["text_raw"].iloc[0][:150])

Calculation of prompt diphoton production cross sections at Tevatron and
  LHC energies   A fully differential calculation in perturbative quantum chr


In [13]:
metadata = {
    "title": df.iloc[0]["title"],
    "authors": df.iloc[0]["authors"],
    "category": df.iloc[0]["category"]
}

print(metadata)

{'title': 'Calculation of prompt diphoton production cross sections at Tevatron and\n  LHC energies', 'authors': "C. Bal\\'azs, E. L. Berger, P. M. Nadolsky, C.-P. Yuan", 'category': 'hep-ph'}


In [14]:
ids = df["id"].tolist()

In [15]:
documents = df["text_raw"].tolist()

In [16]:
type(embeddings)

numpy.ndarray

In [17]:
List[List[float]]

NameError: name 'List' is not defined

In [18]:
embedding_list = embeddings.tolist()

In [19]:
type(embedding_list)

list

In [20]:
metadatas = df[
    [
        "title",
        "authors",
        "category"
    ]
].to_dict("records")

In [21]:
print(metadatas[0])

{'title': 'Calculation of prompt diphoton production cross sections at Tevatron and\n  LHC energies', 'authors': "C. Bal\\'azs, E. L. Berger, P. M. Nadolsky, C.-P. Yuan", 'category': 'hep-ph'}


In [22]:
print(len(ids))
print(len(documents))
print(len(embedding_list))
print(len(metadatas))

1000
1000
1000
1000


In [23]:
collection.add(
    ids=ids,
    embeddings=embedding_list,
    documents=documents,
    metadatas=metadatas
)

ValueError: Expected ID to be a str, got 704.0001 in add.

In [24]:
print(type(df["id"].iloc[0]))
print(type(ids[0]))
print(ids[0])

<class 'numpy.float64'>
<class 'float'>
704.0001


In [25]:
df["id"] = df["id"].astype(str)

In [26]:
print(type(df["id"].iloc[0]))
print(df["id"].iloc[0])

<class 'str'>
704.0001


In [27]:
ids = df["id"].tolist()

In [28]:
print(type(ids[0]))
print(ids[0])

<class 'str'>
704.0001


In [29]:
collection.add(
    ids=ids,
    embeddings=embedding_list,
    documents=documents,
    metadatas=metadatas
)

In [30]:
print(type(df["id"].iloc[0]))
print(type(ids[0]))
print(ids[0])

<class 'str'>
<class 'str'>
704.0001


In [31]:
print(collection.count())

1000


In [32]:
query = "Graph Neural Networks"

query_embedding = model.encode(
    query,
    convert_to_numpy=True
)

In [33]:
print(query_embedding.shape)

(384,)


In [34]:
results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5
)

In [35]:
collection.query()

ValueError: At least one of one of embeddings, documents, images, uris must be provided in query.

In [36]:
results.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances'])

In [37]:
print(results["ids"])

[['704.0598', '704.0686', '704.0648', '704.0345', '704.0392']]


In [38]:
print(results["metadatas"])

[[{'authors': 'Ignazio Licata, Luigi Lella', 'category': 'physics.gen-ph', 'title': 'Evolutionary Neural Gas (ENG): A Model of Self Organizing Network from\n  Input Categorization'}, {'category': 'physics.soc-ph', 'authors': 'Antoniou Ioannis, Tsompa Eleni', 'title': 'Statistical analysis of weighted networks'}, {'title': 'Behavioral response to strong aversive stimuli: A neurodynamical model', 'category': 'q-bio.NC', 'authors': 'Kaushik Majumdar'}, {'category': 'physics.soc-ph', 'title': 'A High Robustness and Low Cost Model for Cascading Failures', 'authors': 'Bing Wang, Beom Jun Kim'}, {'category': 'q-bio.NC', 'authors': 'Marcus Kaiser, Robert Martin, Peter Andras and Malcolm P. Young', 'title': 'Simulation of Robustness against Lesions of Cortical Networks'}]]


In [39]:
for i, paper in enumerate(results["metadatas"][0], start=1):

    print("="*80)

    print(f"Paper {i}")

    print("Title :", paper["title"])

    print("Authors :", paper["authors"])

    print("Category :", paper["category"])

    print()

Paper 1
Title : Evolutionary Neural Gas (ENG): A Model of Self Organizing Network from
  Input Categorization
Authors : Ignazio Licata, Luigi Lella
Category : physics.gen-ph

Paper 2
Title : Statistical analysis of weighted networks
Authors : Antoniou Ioannis, Tsompa Eleni
Category : physics.soc-ph

Paper 3
Title : Behavioral response to strong aversive stimuli: A neurodynamical model
Authors : Kaushik Majumdar
Category : q-bio.NC

Paper 4
Title : A High Robustness and Low Cost Model for Cascading Failures
Authors : Bing Wang, Beom Jun Kim
Category : physics.soc-ph

Paper 5
Title : Simulation of Robustness against Lesions of Cortical Networks
Authors : Marcus Kaiser, Robert Martin, Peter Andras and Malcolm P. Young
Category : q-bio.NC



In [40]:
results["distances"]

[[1.1571781635284424,
  1.2354451417922974,
  1.2729849815368652,
  1.3166090250015259,
  1.3384759426116943]]

In [41]:
df[
    df["text_raw"].str.contains(
        "graph neural",
        case=False,
        na=False
    )
]

,id,title,authors,category,text_raw,text_ml


In [42]:
df[
    df["text_raw"].str.contains(
        "graph",
        case=False,
        na=False
    )
][["title","category"]].head(10)

,title,category
1,Sparsity-certifying Graph Decompositions,math.CO
9,"Partial cubes: structures, characterizations, ...",math.CO
26,Filling-Factor-Dependent Magnetophonon Resonan...,cond-mat.mes-hall
44,Evolution of solitary waves and undular bores ...,nlin.PS
50,Visualizing Teleportation,physics.ed-ph
54,Potassium intercalation in graphite: A van der...,cond-mat.soft
78,Operator algebras associated with unitary comm...,math.OA
178,Experimental nonclassicality of single-photon-...,quant-ph
181,Huge magneto-crystalline anisotropy of x-ray l...,cond-mat.mtrl-sci
207,Some non-braided fusion categories of rank 3,math.GT


In [49]:
def search_papers(query, top_k=5):
    query_embedding = model.encode(
        query,
        convert_to_numpy=True
    )

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )

    print(f"\nQuery: {query}")
    print("=" * 100)

    for i, (meta, doc, dist) in enumerate(
        zip(
            results["metadatas"][0],
            results["documents"][0],
            results["distances"][0]
        ),
        start=1
    ):
        print("=" * 100)
        print(f"Paper {i}")
        print(f"Title    : {meta['title']}")
        print(f"Authors  : {meta['authors']}")
        print(f"Category : {meta['category']}")
        print(f"Distance : {dist:.4f}")
        print(f"Abstract : {doc[:250]}...")

In [50]:
query = "Graph Neural Networks"

In [51]:
search_papers("Graph Theory")


Query: Graph Theory
Paper 1
Title    : Linkedness and ordered cycles in digraphs
Authors  : Daniela K\"uhn and Deryk Osthus
Category : math.CO
Distance : 1.0310
Abstract : Linkedness and ordered cycles in digraphs   The minimum semi-degree of a digraph D is the minimum of its minimum
outdegree and its minimum indegree. We show that every sufficiently large
digraph D with minimum semi-degree at least n/2 +k-1 is k-linke...
Paper 2
Title    : Sparsity-certifying Graph Decompositions
Authors  : Ileana Streinu and Louis Theran
Category : math.CO
Distance : 1.0594
Abstract : Sparsity-certifying Graph Decompositions   We describe a new algorithm, the $(k,\ell)$-pebble game with colors, and use
it obtain a characterization of the family of $(k,\ell)$-sparse graphs and
algorithmic solutions to a family of problems concernin...
Paper 3
Title    : Mediatic graphs
Authors  : J.-Cl. Falmagne, S. Ovchinnikov
Category : math.CO
Distance : 1.0612
Abstract : Mediatic graphs   Any medium can be repres

In [48]:
search_papers("Black Holes")
search_papers("Quantum Computing")
search_papers("Galaxy Formation")


Query: Black Holes

Paper 1
--------------------------------------------------------------------------------
Title    : Topology Change of Black Holes
Authors  : Daisuke Ida and Masaru Siino
Category : gr-qc
Distance : 0.8287

Paper 2
--------------------------------------------------------------------------------
Title    : The First Law for Boosted Kaluza-Klein Black Holes
Authors  : David Kastor, Sourya Ray and Jennie Traschen
Category : hep-th
Distance : 0.9430

Paper 3
--------------------------------------------------------------------------------
Title    : Hawking radiation of linear dilaton black holes
Authors  : G. Clement, J.C. Fabris and G.T. Marques
Category : gr-qc
Distance : 0.9928

Paper 4
--------------------------------------------------------------------------------
Title    : General Relativity Today
Authors  : Thibault Damour
Category : gr-qc
Distance : 1.0020

Paper 5
--------------------------------------------------------------------------------
Title    : Bina

In [52]:
def build_context(results):
    context = ""

    for i, doc in enumerate(results["documents"][0], start=1):

        context += f"\nPaper {i}\n"
        context += "-" * 60 + "\n"
        context += doc
        context += "\n\n"

    return context

In [53]:
context = build_context(results)

print(context[:1000])


Paper 1
------------------------------------------------------------
Evolutionary Neural Gas (ENG): A Model of Self Organizing Network from
  Input Categorization   Despite their claimed biological plausibility, most self organizing networks
have strict topological constraints and consequently they cannot take into
account a wide range of external stimuli. Furthermore their evolution is
conditioned by deterministic laws which often are not correlated with the
structural parameters and the global status of the network, as it should happen
in a real biological system. In nature the environmental inputs are noise
affected and fuzzy. Which thing sets the problem to investigate the possibility
of emergent behaviour in a not strictly constrained net and subjected to
different inputs. It is here presented a new model of Evolutionary Neural Gas
(ENG) with any topological constraints, trained by probabilistic laws depending
on the local distortion errors and the network dimension. The network 

In [54]:
def build_prompt(question, context):

    prompt = f"""
You are an expert AI Research Assistant.

Answer ONLY using the information provided in the context.

If the answer is not available in the context,
say:

"I couldn't find enough information in the retrieved papers."

-----------------------------
Context
-----------------------------

{context}

-----------------------------
Question
-----------------------------

{question}

-----------------------------
Answer
-----------------------------
"""

    return prompt

In [55]:
question = "Explain Hawking Radiation."

prompt = build_prompt(question, context)

print(prompt[:2000])


You are an expert AI Research Assistant.

Answer ONLY using the information provided in the context.

If the answer is not available in the context,
say:

"I couldn't find enough information in the retrieved papers."

-----------------------------
Context
-----------------------------


Paper 1
------------------------------------------------------------
Evolutionary Neural Gas (ENG): A Model of Self Organizing Network from
  Input Categorization   Despite their claimed biological plausibility, most self organizing networks
have strict topological constraints and consequently they cannot take into
account a wide range of external stimuli. Furthermore their evolution is
conditioned by deterministic laws which often are not correlated with the
structural parameters and the global status of the network, as it should happen
in a real biological system. In nature the environmental inputs are noise
affected and fuzzy. Which thing sets the problem to investigate the possibility
of emergent b

In [56]:
question = "Explain Hawking Radiation."

context = build_context(results)

prompt = build_prompt(question, context)

print(prompt[:2000])


You are an expert AI Research Assistant.

Answer ONLY using the information provided in the context.

If the answer is not available in the context,
say:

"I couldn't find enough information in the retrieved papers."

-----------------------------
Context
-----------------------------


Paper 1
------------------------------------------------------------
Evolutionary Neural Gas (ENG): A Model of Self Organizing Network from
  Input Categorization   Despite their claimed biological plausibility, most self organizing networks
have strict topological constraints and consequently they cannot take into
account a wide range of external stimuli. Furthermore their evolution is
conditioned by deterministic laws which often are not correlated with the
structural parameters and the global status of the network, as it should happen
in a real biological system. In nature the environmental inputs are noise
affected and fuzzy. Which thing sets the problem to investigate the possibility
of emergent b

In [57]:
def build_context(results):

    context = ""

    docs = results["documents"][0]
    metas = results["metadatas"][0]

    for i, (doc, meta) in enumerate(zip(docs, metas), start=1):

        context += f"""
Paper {i}
{"="*80}

Title:
{meta['title']}

Authors:
{meta['authors']}

Category:
{meta['category']}

Abstract:
{doc}

{"-"*100}

"""

    return context

In [58]:
def build_prompt(question, context):

    return f"""
You are an AI Research Assistant.

You must answer the user's question ONLY using the retrieved research paper abstracts provided below.

Instructions:
- Do not use outside knowledge.
- If the answer is not present in the context, clearly say:
  "I couldn't find enough information in the retrieved papers."
- If multiple papers discuss the topic, combine their findings.
- Mention paper titles whenever appropriate.
- Keep the answer concise and scientifically accurate.

==========================
Retrieved Research Papers
==========================

{context}

==========================
User Question
==========================

{question}

==========================
Answer
==========================
"""